# 坦桑调研复算记录

## 结论
最低存款中位数100 TZS，最低提现中位数1,000 TZS；12/17家提现门槛更高。此处复算原材料，不代表实时测试。

## 范围与方法
输入：Research TZ _ Bet.xlsx（12表）与坦桑行业调研数据.xlsx（13表）。两份同源，固定原始19品牌；支付子表17家。金额TZS，未识别时长不填零。extract.py保留原值与单元格，analyze.py生成规范记录。外部官方事实与日期见external-evidence.json。

In [1]:
from pathlib import Path
import json, statistics, sqlite3
base=Path.cwd()
if not (base/'normalized.json').exists(): base=base/'analysis/tanzania_gambling_research_2026_09_08'
data=json.loads((base/'normalized.json').read_text())
len(data['brands']),len(data['payment'])

(19, 17)

## 数据与结果
先复算支付门槛，随后用独立SQL核对人数和范围。

In [2]:
p=data['payment']
result={'最低存款中位数':statistics.median(r['min_deposit'] for r in p),'最低提现中位数':statistics.median(r['min_withdraw'] for r in p),'提现门槛更高':sum(r['min_withdraw']>r['min_deposit'] for r in p),'品牌分母':len(p)}
assert result=={'最低存款中位数':100,'最低提现中位数':1000,'提现门槛更高':12,'品牌分母':17}
result

{'最低存款中位数': 100, '最低提现中位数': 1000, '提现门槛更高': 12, '品牌分母': 17}

In [3]:
con=sqlite3.connect(':memory:')
con.execute('CREATE TABLE payments(brand TEXT, deposit REAL, withdraw REAL)')
con.executemany('INSERT INTO payments VALUES (?,?,?)',[(r['brand'],r['min_deposit'],r['min_withdraw']) for r in p])
check=con.execute('SELECT COUNT(*), SUM(withdraw>deposit), MIN(deposit), MAX(withdraw) FROM payments').fetchone()
assert check==(17,12,100,4000)
check

(17, 12, 100.0, 4000.0)

In [4]:
assert round((7959.40/6413.94-1)*100,1)==24.1
assert round(24628003/62793986*100,1)==39.2
assert sum(r['deposit_seconds'] is None for r in p)==2
f=json.loads((base/'formula_validation.json').read_text())
assert len(f)==265 and all(r['cache_match'] for r in f)
x=json.loads((base/'cross_version_reconciliation.json').read_text())
assert x['compared_fields']==378 and len(x['differences'])==1
{'公式复算':len(f),'同源共同字段':x['compared_fields'],'差异':x['differences']}

{'公式复算': 265, '同源共同字段': 378, '差异': [{'brand': 'WinPrincess Bet', 'field': '品牌行', 'original': 'ONLINE!A18:AB18', 'derived': '加工版ONLINE缺项；Profile补列', 'status': '覆盖差异'}]}

## 使用建议
原材料适合制定产品走查和验证清单。当前限额与活动应看具体官方条款；税收、收入、投注额、订阅数与人数保持分开。HTML和飞书共享artifact.json，图表保留原数据，未用模型评分排序。